# R01 · Reproduce the paper (from raw data, in a fresh session)

Rebuilds **every number the draft paper reports** from the public data and this repository, then checks each
one against `paper/expected_values.yaml` and prints PASS / FAIL.

1. Downloads BRIGHT (post-event SAR + label rasters, Hugging Face) and the instance labels (Zenodo 19216381)
   to `/content`: **nothing is read from earlier runs in Drive**.
2. Validates every tile (tile counts, no-data exclusions).
3. Builds the building table (counts, exclusions, building size in tokens: Table II).
4. Extracts DINOv3 ViT-B/16 features at native and 0.3 m, outline and 3 m dilated pooling, on the development
   and secondary events.
5. Runs the benchmark (leave-one-event-out probes, per-event AUROC, event-level comparisons: Tables III–IV),
   measures the footprint-to-SAR offset per tile (Section VI-A), and reruns the controls + ImageNet baseline
   (as p03), the extra encoders (DINOv2, DINOv3 ViT-L web and satellite; as p04), Gate 1 for C2 (as g01) and the
   texture + random-forest reference (as b02) and the confound controls (as p05).
6. Compares everything with the paper.

* **Start from a fresh session:** Runtime → Disconnect and delete runtime, then Runtime → Change runtime type →
  **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN` (DINOv3 is gated).
* **Time:** about 4.5–5 hours (about 11 compute units); keep the tab open. **Outputs:** this run's folder only (`reproduce_report.csv`, the run, the
  executed notebook). Confirmatory labels stay locked: no number that needs them is checked here.
* GPU features are computed in fp16, which is not bit-exact between runs; AUROC is checked to ±0.005.

In [ ]:
NOTEBOOK_NAME = "r01_reproduce_paper"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU, Hugging Face token, environment
import os, json, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
from damagesar_bench.reproduce import environment
ENV = {**environment(), "commit": COMMIT}
print(json.dumps(ENV, indent=1))

In [ ]:
# 1. Data, downloaded fresh to /content (BRIGHT ~3.4 GB from Hugging Face; instance labels 31 MB from Zenodo)
import zipfile, urllib.request
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if not (os.path.isdir(folder) and len(os.listdir(folder)) > 4000):
        p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
        with zipfile.ZipFile(p) as z:
            z.extractall(ROOT)
    print(name, "->", len(os.listdir(folder)), "files")
LABELS = "/content/zips/cvprw26_trainval_instance_labels.zip"
if not os.path.exists(LABELS):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/19216381/files/cvprw26_trainval_instance_labels.zip?download=1", LABELS)
print("instance labels:", round(os.path.getsize(LABELS) / 1e6, 1), "MB")

In [ ]:
# 2. Tile validation (tile counts, no-data exclusions)
!bench validate --root /content/bright --out $RUN/validate
import pandas as pd
TILES = pd.read_csv(f"{RUN}/validate/event_summary.csv")
print(TILES[["event", "role", "n_tiles", "n_excluded_nodata", "n_problem_tiles"]].to_string(index=False))

In [ ]:
# 3. Building table (counts, exclusions, building size in tokens)
from damagesar_bench.config import load_events, load_bench
from damagesar_bench.data.buildings import build_building_table, summarise
events = load_events("/content/damagesar-bench/configs/events.yaml")
# r01 reproduces the development phase. After the freeze, confirmatory labels stay locked here, so r01 never
# reads them and never runs a comparison on the confirmatory events (only k01 does, once).
from dataclasses import replace as _replace
from datetime import date as _date
events = _replace(events, freeze_date=_date(9999, 12, 31))
cfg = load_bench("/content/damagesar-bench/configs/bench.yaml")
B, stats = build_building_table(LABELS, events, cfg.nodata, bright_root=ROOT, target_res_m=0.3)
BSUM = summarise(B)
BSUM.to_csv(f"{RUN}/buildings_summary.csv", index=False)
print(stats)
print(BSUM[["event", "role", "buildings", "kept", "pct_lt_4_tokens_native", "pct_lt_4_tokens_target"]].to_string(index=False))

In [ ]:
# 4. DINOv3 features: native and 0.3 m, outline and 3 m dilated (development + secondary events)
import gc
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile, write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, resolution_comparisons, runs_table
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])]
RESOLUTIONS, METHODS = [None, 0.3], ["outline", "dilated:3"]
enc = get_encoder("dinov3_vitb16")
ENV["dinov3_weights_revision"] = getattr(enc, "weights_revision", None)
print("DINOv3 weights revision:", ENV["dinov3_weights_revision"])
feats, runs = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, METHODS, RESOLUTIONS,
                                     "/content/features", events, log=print)
FEATS = "/content/features_all.parquet"
write_features(feats, FEATS)
runs_table(runs).to_csv(f"{RUN}/timing.csv", index=False)
display(runs_table(runs))
del feats, use, enc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Benchmark (probes, per-event AUROC, comparisons)
from damagesar_bench.run import run_benchmark
comps = resolution_comparisons("dinov3_vitb16", RESOLUTIONS, METHODS)
out = run_benchmark([FEATS], comps, events, f"{RUN}/bench_run", n_boot=2000, seed=0)
M = pd.read_csv(f"{out}/event_metrics.csv"); C = pd.read_csv(f"{out}/comparisons.csv")
F = pd.read_csv(f"{out}/probe_fits.csv")
print(M.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())

In [ ]:
# 5b. Footprint-to-SAR offsets per tile (Section VI-A; CPU, footprints and SAR only, seed 0)
from damagesar_bench.c2_offsets import event_summary, measure_offsets
EV_OFF = sorted(B.loc[B["role"].isin(["development", "secondary"]), "event"].unique())
OFF = measure_offsets(B, lambda t: read_bright_tile(ROOT, t), EV_OFF, factor=4, max_shift_m=40.0, n_null=3, seed=0)
OSUM = event_summary(OFF, peak_z_min=3.0)
tok = {ev: 16 * sum(read_bright_tile(ROOT, g["tile_id"].iloc[0])[1].__dict__[k] for k in ("px_x_m", "px_y_m")) / 2
       for ev, g in OFF.groupby("event")}
OSUM["token_side_m"] = OSUM["event"].map(tok)
OSUM["median_offset_tokens"] = OSUM["median_offset_confident_m"] / OSUM["token_side_m"]
OSUM.to_csv(f"{RUN}/offsets_by_event.csv", index=False)
print(OSUM.round(3).to_string(index=False))

In [ ]:
# 5c. Controls + ImageNet baseline (Section VI-D; as p03): DINOv3 ring 10 m, ImageNet outline / dilated 3 m /
#     ring 10 m, image-free control; native resolution, development + secondary events
from damagesar_bench.controls.imagefree import image_free_features
from damagesar_bench.run import ComparisonSpec
use = B[B["role"].isin(["development", "secondary"])]
FREE = write_features(image_free_features(use, events), "/content/imagefree.parquet")
enc = get_encoder("dinov3_vitb16")
f, _ = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["ring:10"], [None],
                              "/content/features/dinov3_ring", events, log=print)
DRING = write_features(f, "/content/dinov3_ring.parquet"); del f, enc; gc.collect(); torch.cuda.empty_cache()
enc = get_encoder("imagenet_vitb16")
f, _ = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline", "dilated:3", "ring:10"],
                              [None], "/content/features/imagenet", events, log=print)
IMNET = write_features(f, "/content/imagenet.parquet"); del f, enc, use; gc.collect(); torch.cuda.empty_cache()
D, I = "dinov3_vitb16@native", "imagenet_vitb16@native"
comps3 = [ComparisonSpec((D, "outline"), (I, "outline"), "DINOv3 vs ImageNet (outline)"),
          ComparisonSpec((D, "outline"), (D, "ring10m"), "building vs surroundings (DINOv3)"),
          ComparisonSpec((I, "outline"), (I, "ring10m"), "building vs surroundings (ImageNet)"),
          ComparisonSpec((D, "outline"), ("imagefree", "geometry"), "DINOv3 vs image-free"),
          ComparisonSpec((I, "outline"), ("imagefree", "geometry"), "ImageNet vs image-free"),
          ComparisonSpec((I, "dilated3m"), (I, "outline"), "dilated3m vs outline (ImageNet)")]
out3 = run_benchmark([FEATS, str(DRING), str(IMNET), str(FREE)], comps3, events, f"{RUN}/bench_run_controls",
                     n_boot=2000, seed=0, only_compared=True)
M3 = pd.read_csv(f"{out3}/event_metrics.csv"); C3 = pd.read_csv(f"{out3}/comparisons.csv")
F3 = pd.read_csv(f"{out3}/probe_fits.csv")
# DINOv3 native outline is in both runs: keep the step-5 rows, add the new settings
seen = set(zip(M["encoder"], M["pooling"]))
M = pd.concat([M, M3[[k not in seen for k in zip(M3["encoder"], M3["pooling"])]]], ignore_index=True)
C = pd.concat([C, C3], ignore_index=True); F = pd.concat([F, F3], ignore_index=True)
print(M3.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())

In [ ]:
# 5d. More encoders (Section VI-D; as p04): DINOv2 ViT-B/14, DINOv3 ViT-L/16 web and satellite (fp32), outline,
#     native resolution, development + secondary events. About 75 minutes (each ViT-L about 35 minutes).
use = B[B["role"].isin(["development", "secondary"])]
MORE = {}
for name in ["dinov2_vitb14", "dinov3_vitl16", "dinov3_vitl16_sat"]:
    enc = get_encoder(name)
    ENV[f"{name}_weights_revision"] = getattr(enc, "weights_revision", None)
    f, _ = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline"], [None],
                                  f"/content/features/{name}", events, log=print)
    MORE[name] = str(write_features(f, f"/content/{name}.parquet")); del f, enc; gc.collect(); torch.cuda.empty_cache()
del use
B16, V2 = "dinov3_vitb16@native", "dinov2_vitb14@native"
L16, SAT = "dinov3_vitl16@native", "dinov3_vitl16_sat@native"
comps4 = [ComparisonSpec((SAT, "outline"), (L16, "outline"), "DINOv3-L satellite vs web"),
          ComparisonSpec((L16, "outline"), (B16, "outline"), "DINOv3 ViT-L vs ViT-B (web)"),
          ComparisonSpec((B16, "outline"), (V2, "outline"), "DINOv3 vs DINOv2 (ViT-B)")]
out4 = run_benchmark([FEATS, *MORE.values()], comps4, events, f"{RUN}/bench_run_encoders", n_boot=2000, seed=0,
                     only_compared=True)
M4 = pd.read_csv(f"{out4}/event_metrics.csv"); C4 = pd.read_csv(f"{out4}/comparisons.csv")
F4 = pd.read_csv(f"{out4}/probe_fits.csv")
seen = set(zip(M["encoder"], M["pooling"]))
M = pd.concat([M, M4[[k not in seen for k in zip(M4["encoder"], M4["pooling"])]]], ignore_index=True)
C = pd.concat([C, C4], ignore_index=True); F = pd.concat([F, F4], ignore_index=True)
print(M4.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())

In [ ]:
# 5e. Gate 1 (Section VI-B; as g01): C2 image-matched pooling (footprints moved by the step-5b offsets where
#     peak_z >= 3 and the offset beats the null) and 10 same-distance random-direction draws, DINOv3 ViT-B/16
#     native. About 50 minutes.
import numpy as np
from damagesar_bench.c2_offsets import matched_shifts
from damagesar_bench.stats.gates import gate1
SHIFTS = matched_shifts(OFF, peak_z_min=3.0)
use = B[B["role"].isin(["development", "secondary"])]
enc = get_encoder("dinov3_vitb16")
C2F = []
for g, methods in {"a": ["matched"] + [f"rshift:{k}" for k in range(0, 5)],
                   "b": [f"rshift:{k}" for k in range(5, 10)]}.items():
    f, _ = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, methods, [None],
                                  f"/content/features/c2_{g}", events, log=print, shifts=SHIFTS)
    C2F.append(str(write_features(f, f"/content/dinov3_c2_{g}.parquet"))); del f; gc.collect()
del enc, use; torch.cuda.empty_cache()
comps5 = [ComparisonSpec((D, "matched"), (D, "outline"), "C2 matched vs outline"),
          ComparisonSpec((D, "matched"), (D, "dilated3m"), "C2 matched vs dilated 3 m"),
          ComparisonSpec((D, "matched"), (D, "ring10m"), "C2 matched vs ring 10 m")]
comps5 += [ComparisonSpec((D, f"rshift{k}"), (D, "outline"), f"random shift {k} vs outline") for k in range(10)]
out5 = run_benchmark([FEATS, str(DRING), *C2F], comps5, events, f"{RUN}/bench_run_gate1", n_boot=2000, seed=0,
                     only_compared=True)
M5 = pd.read_csv(f"{out5}/event_metrics.csv"); C5 = pd.read_csv(f"{out5}/comparisons.csv").set_index("label")
F5 = pd.read_csv(f"{out5}/probe_fits.csv")
g1 = gate1(json.loads(C5.loc["C2 matched vs outline", "gains"]),
           [float(np.mean(list(json.loads(C5.loc[f"random shift {k} vs outline", "gains"]).values()))) for k in range(10)])
GATE = {"gate1.mean_gain": round(float(g1.mean_gain), 3), "gate1.n_better": int(g1.n_better),
        "gate1.passed": bool(g1.passed)}
print(GATE, "| null p95:", round(float(g1.null_threshold), 4))
seen = set(zip(M["encoder"], M["pooling"]))
M = pd.concat([M, M5[[p == "matched" and (e, p) not in seen for e, p in zip(M5["encoder"], M5["pooling"])]]],
              ignore_index=True)
C = pd.concat([C, C5.reset_index()[lambda d: d["label"].str.startswith("C2 ")]], ignore_index=True)
F = pd.concat([F, F5], ignore_index=True)
print(M5[M5["pooling"] == "matched"][["event", "auroc"]].round(3).to_string(index=False))

In [ ]:
# 5f. Classical reference (Section VI; as b02): intensity + GLCM texture of each footprint (and a 32 m context
#     square), random forest per held-out event. CPU, deterministic. About 10 minutes.
from damagesar_bench.baselines.texture import leave_one_event_out as texture_loeo, texture_table
from damagesar_bench.stats.compare import compare as compare_settings, event_metrics
use = B[B["role"].isin(["development", "secondary"])]
TX6 = texture_table(use, lambda t: read_bright_tile(ROOT, t), log=print); del use
dev6 = events.keys_with_role("development")
FS6 = pd.concat([texture_loeo(TX6, dev6, p) for p in ("footprint", "footprint_ctx32m")], ignore_index=True)
M6 = event_metrics(FS6, events, n_boot=2000, seed=0)
M = pd.concat([M, M6], ignore_index=True)
TX, TXC = ("texture_rf", "footprint"), ("texture_rf", "footprint_ctx32m")
rows6 = []
for label, a, b in [("texture vs image-free", TX, ("imagefree", "geometry")),
                    ("texture context vs footprint", TXC, TX),
                    ("DINOv3 ViT-B probe vs texture", ("dinov3_vitb16@native", "outline"), TX),
                    ("ImageNet probe vs texture", ("imagenet_vitb16@native", "outline"), TX),
                    ("satellite ViT-L probe vs texture", ("dinov3_vitl16_sat@native", "outline"), TX)]:
    c6 = compare_settings(M, a, b, events=dev6)
    rows6.append({"label": label, "estimate": c6.interval.estimate, "lower": c6.interval.lower,
                  "upper": c6.interval.upper, "verdict": c6.verdict.meaning})
C = pd.concat([C, pd.DataFrame(rows6)], ignore_index=True)
print(M6.pivot_table(index="pooling", columns="event", values="auroc").round(3).to_string())
print(pd.DataFrame(rows6).round(3).to_string(index=False))

In [ ]:
# 5g. Confound controls (Section VI-E; as p05): DINOv3 ViT-B in fp32, web DINOv3 ViT-L with the satellite statistics.
#     About 45 minutes.
use = B[B["role"].isin(["development", "secondary"])]
for name in ["dinov3_vitb16_fp32", "dinov3_vitl16_satnorm"]:
    enc = get_encoder(name)
    ENV[f"{name}_weights_revision"] = getattr(enc, "weights_revision", None)
    f, _ = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline"], [None],
                                  f"/content/features/{name}", events, log=print)
    MORE[name] = str(write_features(f, f"/content/{name}.parquet")); del f, enc; gc.collect(); torch.cuda.empty_cache()
del use
B32, LSN = "dinov3_vitb16_fp32@native", "dinov3_vitl16_satnorm@native"
comps7 = [ComparisonSpec((B32, "outline"), (B16, "outline"), "DINOv3 ViT-B: fp32 vs fp16"),
          ComparisonSpec((LSN, "outline"), (L16, "outline"), "DINOv3 ViT-L web: satellite vs ImageNet statistics"),
          ComparisonSpec((SAT, "outline"), (LSN, "outline"), "DINOv3 ViT-L: satellite vs web, same statistics"),
          ComparisonSpec((L16, "outline"), (B32, "outline"), "DINOv3 web, fp32: ViT-L vs ViT-B")]
out7 = run_benchmark([FEATS, MORE["dinov3_vitl16"], MORE["dinov3_vitl16_sat"], MORE["dinov3_vitb16_fp32"],
                      MORE["dinov3_vitl16_satnorm"]], comps7, events, f"{RUN}/bench_run_confounds", n_boot=2000,
                     seed=0, only_compared=True)
M7 = pd.read_csv(f"{out7}/event_metrics.csv"); C7 = pd.read_csv(f"{out7}/comparisons.csv")
F7 = pd.read_csv(f"{out7}/probe_fits.csv")
seen = set(zip(M["encoder"], M["pooling"]))
M = pd.concat([M, M7[[k not in seen for k in zip(M7["encoder"], M7["pooling"])]]], ignore_index=True)
C = pd.concat([C, C7], ignore_index=True); F = pd.concat([F, F7], ignore_index=True)
print(M7.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())

In [ ]:
# 6. Compare with the paper
from damagesar_bench.reproduce import collect_actual, compare, load_expected, verdict
exp = load_expected("/content/damagesar-bench/paper/expected_values.yaml")
got = collect_actual(TILES, B, BSUM, M, C, F, offsets=OSUM)
got.update(GATE)
REPORT = compare(exp, got)
REPORT.to_csv(f"{RUN}/reproduce_report.csv", index=False)
with open(f"{RUN}/environment.json", "w") as fh:
    json.dump(ENV, fh, indent=1)
pd.set_option("display.max_rows", 200); pd.set_option("display.width", 200)
display(REPORT)
bad = REPORT[REPORT["status"].isin(["FAIL", "MISSING"])]
if len(bad):
    print("\nNot matching:"); print(bad.to_string(index=False))
print("\n" + verdict(REPORT))

In [ ]:
os.makedirs(f"{PROJECT}/03_notebooks/reproduce", exist_ok=True)

# File the executed notebook (with outputs) into this run's folder and 03_notebooks/reproduce/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/reproduce/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))